In [ ]:
# Run from the repository root, or open this notebook anywhere inside it.
from pathlib import Path
import os
_repo_candidates = [Path.cwd(), *Path.cwd().parents]
_project_root = next((p for p in _repo_candidates if (p / "code/data_process").is_dir() and (p / "README.md").is_file()), None)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the repository.")
os.chdir(_project_root)


In [ ]:
import pandas as pd
import os

# Define the base path
base_path = "./data-all-waves"
file_path = os.path.join(base_path, "update_merged_nscg_2010_to_2021.csv")
df = pd.read_csv(file_path)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
df.columns

In [ ]:
# Print original number of observations
print(f"Original number of observations: {len(df)}")

# Drop observations where BAYR is missing or equals 9998.0
df = df.dropna(subset=['BAYR'])  # Drop missing values
df = df[df['BAYR'] != 9998.0]    # Drop observations with BAYR = 9998.0

# Print final number of observations
print(f"Number of observations after dropping: {len(df)}")

In [ ]:
# List of variables to check
matching_vars = ['BIRYR', 'GENDER', 'RACEM', 'BAYR', 'BAMN', 'COHORT', 
                'BTHUS', 'BTHRGN', 'HSYR', 'EDDAD', 'EDMOM']

# 1. Check missing values for each variable
print("Missing Values Analysis:")
print("------------------------")
missing_analysis = df[matching_vars].isnull().sum()
missing_pct = (df[matching_vars].isnull().sum() / len(df)) * 100
missing_summary = pd.DataFrame({
    'Missing Count': missing_analysis,
    'Missing Percentage': missing_pct.round(2)
})
print(missing_summary)

# 2. Check unique values for each variable by REFYR
print("\nUnique Values Analysis by REFYR:")
print("--------------------------------")
for var in matching_vars:
    print(f"\nVariable: {var}")
    print("-" * (len(var) + 10))
    
    # Create a summary of unique values by REFYR
    unique_vals_by_year = {}
    for year in df['REFYR'].unique():
        # Get unique values without sorting
        unique_vals = df[df['REFYR']==year][var].unique()
        
        # Convert to list for consistent handling
        unique_vals = list(unique_vals)
        
        unique_vals_by_year[year] = unique_vals
        print(f"\nREFYR {year}:")
        print(f"Number of unique values: {len(unique_vals)}")
        
        # Print first few values as example and data type
        print(f"Sample values: {unique_vals[:5]}")
        print(f"Data type: {df[df['REFYR']==year][var].dtype}")
        
        # If numeric, show range
        if pd.api.types.is_numeric_dtype(df[df['REFYR']==year][var]):
            valid_vals = df[df['REFYR']==year][var].dropna()
            if len(valid_vals) > 0:
                print(f"Range: {valid_vals.min()} to {valid_vals.max()}")
    
    # Check for data type consistency across years
    dtypes = {year: df[df['REFYR']==year][var].dtype 
              for year in df['REFYR'].unique()}
    if len(set(dtypes.values())) > 1:
        print(f"\nWarning: Inconsistent data types across years:")
        for year, dtype in dtypes.items():
            print(f"Year {year}: {dtype}")

### Clean EDMOM, EDDAD

In [ ]:
# First let's see current unique values in 2010
print("Before cleaning - unique values in 2010:")
print(df[df['REFYR']==2010]['EDMOM'].unique())
print(df[df['REFYR']==2010]['EDDAD'].unique())

# Function to convert values to integers but keep 'E'
def clean_ed(x):
    if pd.isna(x):  # Keep NaN as is
        return x
    if isinstance(x, str):
        if x.isdigit():  # If it's a string number
            return int(x)
        elif x == 'E':   # Keep 'E' as is
            return 'E'
    return x  # Return original value if it's already an integer

# Clean EDMOM for 2010
mask_2010 = df['REFYR'] == 2010
df.loc[mask_2010, 'EDMOM'] = df.loc[mask_2010, 'EDMOM'].apply(clean_ed)
# Clean EDDAD for 2010
df.loc[mask_2010, 'EDDAD'] = df.loc[mask_2010, 'EDDAD'].apply(clean_ed)

# Verify all years now
for year in sorted(df['REFYR'].unique()):
    print(f"\nREFYR {year}:")
    print(sorted([x for x in df[df['REFYR']==year]['EDMOM'].unique() if x != 'E']) + (['E'] if 'E' in df[df['REFYR']==year]['EDMOM'].unique() else []))
    print(sorted([x for x in df[df['REFYR']==year]['EDDAD'].unique() if x != 'E']) + (['E'] if 'E' in df[df['REFYR']==year]['EDDAD'].unique() else []))
   

# Matching 2010 and 2013
now, let's try matching for REFYR 2010 and 2013 using the following criteria: 
1. BIRYR within 1 year error
1. exact match for GENDER, RACEM, BTHUS, BAYR, BAMN and COHORT




In [ ]:
# Separate 2010 and 2013 data
df_2010 = df[df['REFYR'] == 2010].copy()
df_2013 = df[df['REFYR'] == 2013].copy()

print(f"Number of observations in 2010: {len(df_2010)}")
print(f"Number of observations in 2013: {len(df_2013)}")

# First find exact matches on all variables except BIRYR
exact_match_vars = ['GENDER', 'RACEM', 'BTHUS', 'BAYR', 'BAMN', 'COHORT']

# Merge based on exact matches
matches = pd.merge(df_2010, df_2013, on=exact_match_vars, suffixes=('_2010', '_2013'))

# Filter for BIRYR within 1 year
matches = matches[abs(matches['BIRYR_2010'] - matches['BIRYR_2013']) <= 1]

# Print summary statistics
print("\nMatching Summary:")
print("-----------------")
print(f"Total number of matches found: {len(matches)}")
print(f"Number of unique 2010 observations matched: {matches['REFID_2010'].nunique()}")
print(f"Number of unique 2013 observations matched: {matches['REFID_2013'].nunique()}")

# Distribution of BIRYR differences
print("\nDistribution of BIRYR differences:")
print((matches['BIRYR_2013'] - matches['BIRYR_2010']).value_counts().sort_index())

## Reverse way

## look into potential matching variables

In [ ]:
# Get 2010 and 2013 data
df_2010 = df[df['REFYR'] == 2010]
df_2013 = df[df['REFYR'] == 2013]

# List of variables to check
vars_to_check = ['BIRYR', 'GENDER', 'RACEM', 'BAYR', 'BAMN', 'COHORT', 
                 'BTHUS', 'BTHRGN', 'HSYR', 'EDDAD', 'EDMOM', 'NBAMEMG', 'NBAMENG',
                 'NBAMED','HSST','BTHST']

# Find matches based on REFID
matches = pd.merge(
    df_2010[['REFID'] + vars_to_check], 
    df_2013[['REFID'] + vars_to_check], 
    on='REFID',
    suffixes=('_2010', '_2013')
)

print(f"Total number of matched observations between 2010 and 2013: {len(matches)}")
print("\nConsistency rates and inconsistent examples:")
print("-" * 50)

# Check consistency for each variable
for var in vars_to_check:
    # Calculate consistency rate
    consistent = (matches[f'{var}_2010'] == matches[f'{var}_2013']).sum()
    rate = (consistent / len(matches) * 100)
    print(f"\n{var}: {rate:.1f}%")
        
    # Show examples of inconsistent cases if any exist
    inconsistent = matches[matches[f'{var}_2010'] != matches[f'{var}_2013']]
    if len(inconsistent) > 0:
        print(f"Examples of inconsistent {var} values (first 5):")
        inconsistent_examples = pd.DataFrame({
            f'{var}_2010': inconsistent[f'{var}_2010'],
            f'{var}_2013': inconsistent[f'{var}_2013']
        }).head()
        print(inconsistent_examples)

In [ ]:
matches

In [ ]:
BTHST_unmatched = matches[matches['BTHST_2010'] != matches['BTHST_2013']]

BTHST_unmatched

In [ ]:
HSST_unmatched = matches[matches['HSST_2010'] != matches['HSST_2013']]
HSST_unmatched


In [ ]:
RACEM_unmatched = matches[matches['RACEM_2010'] != matches['RACEM_2013']]
RACEM_unmatched

### *** matching criteria *** evaluate 2010 and 2013 (w/o birth year condition)

In [ ]:
# Define the variable list
vars_list_1 = ['GENDER', 'RACEM', 'BAYR', 'BAMN', 'COHORT', 
                 'BTHUS', 'BTHRGN', 'HSYR', 'EDDAD', 'EDMOM', 'NBAMEMG', 'NBAMENG',
                 'NBAMED','HSST','BTHST']


# Get 2010 and 2013 data
df_2010 = df[df['REFYR'] == 2010]
df_2013 = df[df['REFYR'] == 2013]

# Find matches using these variables
our_matches = pd.merge(
    df_2010[['REFID'] + vars_list_1], 
    df_2013[['REFID'] + vars_list_1], 
    on=vars_list_1,
    suffixes=('_2010', '_2013')
)

# Get actual matches based on REFID
actual_matches = pd.merge(
    df_2010[['REFID']], 
    df_2013[['REFID']], 
    on='REFID'
)

# Analysis of matching results
print("\nMatching Results Analysis:")
print("-" * 50)
print(f"Total observations in 2010: {len(df_2010)}")
print(f"Total observations in 2013: {len(df_2013)}")
print(f"Number of matches found by our method: {len(our_matches)}")
print(f"Number of actual matches (based on REFID): {len(actual_matches)}")

# Analyze accuracy
# 1. Correct matches (our matches that are actual matches)
correct_matches = our_matches[our_matches['REFID_2010'] == our_matches['REFID_2013']]

# 2. Wrong matches (our matches that are not actual matches)
wrong_matches = our_matches[our_matches['REFID_2010'] != our_matches['REFID_2013']]

# 3. Missed matches (actual matches we didn't find)
found_refids = set(correct_matches['REFID_2010'])
actual_refids = set(actual_matches['REFID'])
missed_refids = actual_refids - found_refids

print("\nAccuracy Analysis:")
print("-" * 50)
print(f"Correct matches: {len(correct_matches)}")
print(f"Wrong matches: {len(wrong_matches)}")
print(f"Missed matches: {len(missed_refids)}")
print(f"Precision (correct/total matches found): {len(correct_matches)/len(our_matches)*100:.1f}%")
print(f"Recall (correct/actual matches): {len(correct_matches)/len(actual_matches)*100:.1f}%")

### *** matching criteria *** evaluate 2010 and 2013 (+ birth year condition)

In [ ]:
# Define the variable list
vars_list_1 = ['GENDER','BAYR', 'BAMN', 'COHORT','BTHUS', 'BTHRGN', 'HSYR', 'EDDAD', 'EDMOM','NBAMEMG', 'NBAMENG',
                 'NBAMED']
vars_list_1 = ['GENDER', 'RACEM', 'BAYR', 'BAMN', 'COHORT', 
                 'BTHUS', 'BTHRGN', 'HSYR', 'EDDAD', 'EDMOM', 'NBAMEMG', 'NBAMENG',
                 'NBAMED','HSST','BTHST']

# Get 2010 and 2013 data
df_2010 = df[df['REFYR'] == 2010]
df_2013 = df[df['REFYR'] == 2013]

# First merge on exact matches for vars_list_1
# Then filter for BIRYR within 1 year difference
our_matches = pd.merge(
    df_2010[['REFID', 'BIRYR'] + vars_list_1], 
    df_2013[['REFID', 'BIRYR'] + vars_list_1], 
    on=vars_list_1,
    suffixes=('_2010', '_2013')
)

# Add BIRYR filter
our_matches = our_matches[abs(our_matches['BIRYR_2010'] - our_matches['BIRYR_2013']) <= 1]

# Get actual matches based on REFID
actual_matches = pd.merge(
    df_2010[['REFID']], 
    df_2013[['REFID']], 
    on='REFID'
)

# Analysis of matching results
print("\nMatching Results Analysis:")
print("-" * 50)
print(f"Total observations in 2010: {len(df_2010)}")
print(f"Total observations in 2013: {len(df_2013)}")
print(f"Number of matches found by our method: {len(our_matches)}")
print(f"Number of actual matches (based on REFID): {len(actual_matches)}")

# Analyze accuracy
# 1. Correct matches (our matches that are actual matches)
correct_matches = our_matches[our_matches['REFID_2010'] == our_matches['REFID_2013']]

# 2. Wrong matches (our matches that are not actual matches)
wrong_matches = our_matches[our_matches['REFID_2010'] != our_matches['REFID_2013']]

# 3. Missed matches (actual matches we didn't find)
found_refids = set(correct_matches['REFID_2010'])
actual_refids = set(actual_matches['REFID'])
missed_refids = actual_refids - found_refids

print("\nAccuracy Analysis:")
print("-" * 50)
print(f"Correct matches: {len(correct_matches)}")
print(f"Wrong matches: {len(wrong_matches)}")
print(f"Missed matches: {len(missed_refids)}")
print(f"Precision (correct/total matches found): {len(correct_matches)/len(our_matches)*100:.1f}%")
print(f"Recall (correct/actual matches): {len(correct_matches)/len(actual_matches)*100:.1f}%")

# Additional analysis of BIRYR differences in correct matches
print("\nBIRYR Difference Analysis in Correct Matches:")
print("-" * 50)
biryr_diff = correct_matches['BIRYR_2013'] - correct_matches['BIRYR_2010']
print(biryr_diff.value_counts().sort_index())

In [ ]:
wrong_matches

# Try matching including all waves: use 'match_key'

In [ ]:
vars_list_1 = ['GENDER', 'RACEM', 'BAYR', 'BAMN', 'COHORT', 
                 'BTHUS', 'BTHRGN', 'HSYR', 'EDDAD', 'EDMOM', 'NBAMEMG', 'NBAMENG',
                 'NBAMED','HSST','BTHST']

In [ ]:
# Step 1: Create a matching key that combines all exact matching variables
print("Creating matching keys...")
df['match_key'] = (df[vars_list_1]
                   .astype(str)
                   .agg('_'.join, axis=1))

# Step 2: Find potential matches based on match_key
print("Finding potential matches...")
potential_matches = (df.groupby('match_key')
                      .filter(lambda x: len(x) > 1))  # Only keep groups with potential matches

# Step 3: Within each match_key group, check BIRYR condition
print("Checking BIRYR condition...")
matches = []
for key, group in potential_matches.groupby('match_key'):
    # Get all pairs of years in this group
    years_in_group = sorted(group['REFYR'].unique())
    
    # For each pair of years
    for i, year1 in enumerate(years_in_group):
        for year2 in years_in_group[i+1:]:
            year1_data = group[group['REFYR'] == year1]
            year2_data = group[group['REFYR'] == year2]
            
            # Check BIRYR condition for all combinations
            for _, row1 in year1_data.iterrows():
                for _, row2 in year2_data.iterrows():
                    if abs(row1['BIRYR'] - row2['BIRYR']) <= 1:
                        matches.append({
                            'match_key': key,
                            'year1': year1,
                            'year2': year2,
                            'refid1': row1['REFID'],
                            'refid2': row2['REFID'],
                            'biryr1': row1['BIRYR'],
                            'biryr2': row2['BIRYR']
                        })

# Convert matches to DataFrame
matches_df = pd.DataFrame(matches)

print("\nInitial Matching Summary:")
print(f"Total number of matches found: {len(matches_df)}")
print("\nMatches by year pairs:")
year_pairs = matches_df.groupby(['year1', 'year2']).size()
print(year_pairs)

# We can now analyze these matches further:
# 1. Find multiple matches
# 2. Assign IDs to matched groups
# 3. Calculate statistics


In [ ]:
matches_df

In [ ]:
# First, let's look at matches between 2010 and 2013 only
print("Analyzing 2010-2013 matches...")
matches_10_13 = matches_df[
    (matches_df['year1'] == 2010) & 
    (matches_df['year2'] == 2013)
]

print(f"\nNumber of matches between 2010-2013: {len(matches_10_13)}")

# Check multiple matches for 2010-2013
multiple_2010 = matches_10_13.groupby('refid1')['refid2'].count()
multiple_2013 = matches_10_13.groupby('refid2')['refid1'].count()

print("\n2010 observations with multiple matches in 2013:")
print(multiple_2010[multiple_2010 > 1].value_counts())
print("\n2013 observations with multiple matches to 2010:")
print(multiple_2013[multiple_2013 > 1].value_counts())

# Now look at matches involving other years
print("\nAnalyzing matches involving other years...")
other_matches = matches_df[
    ~((matches_df['year1'] == 2010) & (matches_df['year2'] == 2013))
]

# Group by year pairs to see match patterns
year_pair_counts = other_matches.groupby(['year1', 'year2']).size()
print("\nNumber of matches by year pairs:")
print(year_pair_counts)

# For each year, count how many potential matches an observation has
for year in sorted(set(matches_df['year1'].unique()) | set(matches_df['year2'].unique())):
    if year not in [2010, 2013]:  # Skip 2010 and 2013 as we already analyzed them
        print(f"\nAnalyzing year {year}:")
        # Count matches as year1
        matches_as_year1 = matches_df[matches_df['year1'] == year]
        matches_count1 = matches_as_year1.groupby('match_key').size()
        
        # Count matches as year2
        matches_as_year2 = matches_df[matches_df['year2'] == year]
        matches_count2 = matches_as_year2.groupby('match_key').size()
        
        print(f"As earlier year (year1) - distribution of number of matches:")
        print(matches_count1.value_counts().sort_index())
        print(f"As later year (year2) - distribution of number of matches:")
        print(matches_count2.value_counts().sort_index())

In [ ]:
# First, identify duplicate match_keys within years
print("Identifying duplicate match_keys within years...")
duplicate_flags = {}

for year in sorted(df['REFYR'].unique()):
    year_data = df[df['REFYR'] == year]
    duplicate_keys = year_data['match_key'].value_counts()
    
    # Store match_keys that have duplicates within this year
    duplicate_flags[year] = duplicate_keys[duplicate_keys > 1].index.tolist()

# Add this information to our matches dataframe
print("\nFlagging matches involving within-year duplicates...")
matches_df['has_duplicates'] = matches_df.apply(
    lambda row: (row['match_key'] in duplicate_flags[row['year1']]) or 
                (row['match_key'] in duplicate_flags[row['year2']]), 
    axis=1
)

# Print summary of flagged matches
print("\nSummary of flagged matches:")
print(f"Total matches: {len(matches_df)}")
print(f"Matches involving within-year duplicates: {matches_df['has_duplicates'].sum()}")
print(f"Clean matches: {(~matches_df['has_duplicates']).sum()}")

# Look at example of a flagged case
print("\nExample of a match involving within-year duplicates:")
example_flagged = matches_df[matches_df['has_duplicates']].iloc[0]
print("\nMatch details:")
print(example_flagged)

print("\nOriginal observations involved:")
key = example_flagged['match_key']
year1 = example_flagged['year1']
year2 = example_flagged['year2']
print(f"\nYear {year1} observations:")
print(df[(df['REFYR']==year1) & (df['match_key']==key)][['REFYR'] + vars_list_1 + ['BIRYR']])
print(f"\nYear {year2} observations:")
print(df[(df['REFYR']==year2) & (df['match_key']==key)][['REFYR'] + vars_list_1 + ['BIRYR']])

## assign ids to "clean" matches (without multiple matched observations within a wave)

In [ ]:
# Separate clean matches and matches with duplicates
clean_matches = matches_df[~matches_df['has_duplicates']].copy()
duplicate_matches = matches_df[matches_df['has_duplicates']].copy()

def assign_ids_clean_efficient(clean_matches_df, df):
    print("Creating index mapping...")
    # Create index mapping once
    index_mapping = {}
    for year in sorted(df['REFYR'].unique()):
        # Create mapping from (year, match_key) to index for faster lookup
        year_data = df[df['REFYR'] == year]
        index_mapping.update({
            (year, mk): idx for mk, idx in 
            zip(year_data['match_key'], year_data.index)
        })
    
    print("Creating connected components...")
    # Create graph for connected components
    from collections import defaultdict
    graph = defaultdict(set)
    
    # Add edges to graph
    for _, match in clean_matches_df.iterrows():
        idx1 = index_mapping[(match['year1'], match['match_key'])]
        idx2 = index_mapping[(match['year2'], match['match_key'])]
        graph[idx1].add(idx2)
        graph[idx2].add(idx1)
    
    # Find connected components
    def find_components(graph):
        components = []
        seen = set()
        
        def dfs(node, component):
            seen.add(node)
            component.add(node)
            for neighbor in graph[node]:
                if neighbor not in seen:
                    dfs(neighbor, component)
        
        for node in graph:
            if node not in seen:
                component = set()
                dfs(node, component)
                components.append(component)
        
        return components
    
    print("Assigning IDs...")
    # Each component gets one ID
    id_mapping = {}
    for i, component in enumerate(find_components(graph), 1):
        for idx in component:
            id_mapping[idx] = i
    
    return id_mapping, i + 1

print("Starting ID assignment...")
clean_id_mapping, next_id = assign_ids_clean_efficient(clean_matches, df)

# Print summary statistics
print("\nClean matches ID assignment summary:")
print(f"Number of unique IDs assigned: {len(set(clean_id_mapping.values()))}")
print(f"Number of observations assigned IDs: {len(clean_id_mapping)}")

# Update the dataframe
df['assigned_id'] = pd.Series(clean_id_mapping)

### verify assignments using 2010 and 2013 refids

In [ ]:
import numpy as np
# Get 2010 and 2013 observations with assigned_ids
verify_df = df[df['REFYR'].isin([2010, 2013]) & df['assigned_id'].notna()]

# Group by assigned_id and check REFIDs
id_verification = verify_df.groupby('assigned_id').agg({
    'REFYR': 'nunique',
    'REFID': lambda x: len(set(x) - {None, np.nan})
}).reset_index()

print("\nVerification Summary:")
print("-" * 50)
print(f"Total number of assigned_ids appearing in 2010/2013: {len(id_verification)}")

# Classify matches
correct_matches = id_verification[(id_verification['REFYR'] == 2) & (id_verification['REFID'] == 1)]
wrong_matches = id_verification[(id_verification['REFYR'] == 2) & (id_verification['REFID'] > 1)]
single_year = id_verification[id_verification['REFYR'] == 1]

print(f"\nCorrect matches (same REFID across years): {len(correct_matches)}")
print(f"Wrong matches (different REFIDs): {len(wrong_matches)}")
print(f"Single year appearances: {len(single_year)}")

# Look at some wrong matches in detail
if len(wrong_matches) > 0:
    print("\nExample of wrong matches:")
    wrong_ids = wrong_matches['assigned_id'].head()
    for wrong_id in wrong_ids:
        print(f"\nassigned_id: {wrong_id}")
        print(verify_df[verify_df['assigned_id'] == wrong_id][['REFYR', 'REFID', 'match_key', 'BIRYR']])

In [ ]:
id_verification

In [ ]:
df[df['assigned_id'] == 37568.0]

### look into NaN assigned ids -> include unmatched obs, and multiple matched obs within year

In [ ]:
# Check total NaN count in assigned_id
total_nans = df['assigned_id'].isna().sum()
print(f"Total number of NaN values in assigned_id: {total_nans}")
print(f"Percentage of NaN values: {(total_nans/len(df))*100:.2f}%")

# Check NaN distribution by year
print("\nNaN distribution by year:")
for year in sorted(df['REFYR'].unique()):
    year_data = df[df['REFYR'] == year]
    year_nans = year_data['assigned_id'].isna().sum()
    print(f"Year {year}:")
    print(f"Number of NaNs: {year_nans}")
    print(f"Percentage of NaNs: {(year_nans/len(year_data))*100:.2f}%")
    print()

In [ ]:
# For observations with NaN assigned_id
unmatched = df[df['assigned_id'].isna()]

# Check how many have unique match_keys (no potential matches at all)
unique_keys = df.groupby('match_key').size()
solo_keys = unique_keys[unique_keys == 1].index

print("Analysis of unmatched observations:")
print(f"Total unmatched observations: {len(unmatched)}")
print(f"Observations with unique match_keys: {len(unmatched[unmatched['match_key'].isin(solo_keys)])}")

# For those with shared match_keys but still unmatched
shared_but_unmatched = unmatched[~unmatched['match_key'].isin(solo_keys)]
print(f"Observations with shared match_keys but unmatched: {len(shared_but_unmatched)}")

# Look at an example of shared but unmatched
if len(shared_but_unmatched) > 0:
    example_key = shared_but_unmatched['match_key'].iloc[0]
    print("\nExample of shared match_key but unmatched:")
    print(df[df['match_key'] == example_key][['REFYR', 'BIRYR', 'assigned_id']].sort_values('REFYR'))

In [ ]:
# Find observations with unique match_keys
unique_keys = df.groupby('match_key').size()
solo_keys = unique_keys[unique_keys == 1].index

# Get current max assigned_id to continue from there
current_max_id = df['assigned_id'].max()
if pd.isna(current_max_id):
   current_max_id = 0

# Assign new IDs to solo observations
new_ids = range(int(current_max_id) + 1, int(current_max_id) + 1 + len(solo_keys))
solo_id_mapping = dict(zip(solo_keys, new_ids))

# Update assigned_id for solo observations
df.loc[df['match_key'].isin(solo_keys), 'assigned_id'] = df.loc[df['match_key'].isin(solo_keys), 'match_key'].map(solo_id_mapping)

# Print results
print(f"Number of unique match_keys: {len(solo_keys)}")
print(f"New assigned_id range: {int(current_max_id) + 1} to {int(current_max_id) + len(solo_keys)}")
print(f"Remaining NaNs in assigned_id: {df['assigned_id'].isna().sum()}")

In [ ]:
df.columns

In [ ]:
df_2010 = df[df['REFYR']== 2010]

In [ ]:
df_2010[['REFYR', 'REFID', 'assigned_id']]

In [ ]:
df_2010 = df[df['REFYR'] == 2010].copy()
df_2013 = df[df['REFYR'] == 2013].copy()

In [ ]:
check_matches = pd.merge(df_2010, df_2013, on= 'REFID', suffixes=('_2010', '_2013'))

In [ ]:
check_matches[['REFID','REFYR_2010', 'assigned_id_2010', 'REFYR_2013', 'assigned_id_2013']]

In [ ]:
df_2010[df_2010['REFID'] == 'D09651744']

In [ ]:
df_2013[df_2013['REFID'] == 'D09651744']

In [ ]:
# Save to csv
df.to_csv('data-all-waves/matched_df.csv', index=False)

# Verify the save
print(f"File saved with {len(df)} rows and {len(df.columns)} columns")